# Cobertura e qualidade do dump Argus

## Escopo e fuso horário

Os timestamps apresentados por este notebook estão em **UTC**. O BRT (`America/Sao_Paulo`) deve ser aplicado apenas na visualização, nunca como interpretação implícita dos dados de origem.

O objetivo aqui é medir cobertura, qualidade e ritmo de chegada dos dados. Por isso, a janela usada em cada gráfico é derivada da cobertura real observada no dump, e não de datas arbitrárias.

In [13]:
from dump_explorer.charts import (
    cadence_histogram,
    collector_timeline,
    coverage_heatmap,
    missingness_bar,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.quality import (
    load_collector_health,
    load_missingness,
    load_series_cadence,
    load_source_coverage,
    load_source_daily_coverage,
    load_table_coverage,
)
from dump_explorer.signals import list_series, load_source_distribution
import pandas as pd

settings = load_settings()
engine = create_db_engine(settings)

## Janela real dos dados

In [14]:
s = load_source_coverage(engine)           # rápido; retorna first_ts/last_ts em UTC
if s.empty or s['first_ts'].dropna().empty:
    raise RuntimeError("source_coverage vazio — verifique conexão/importe o dump")
start_ms = int(s['first_ts'].dropna().min().value // 10**6)
end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
print("start_ms=", start_ms, "end_ms=", end_ms)
print("start (UTC) =", pd.to_datetime(start_ms, unit='ms', utc=True))
print("end   (UTC) =", pd.to_datetime(end_ms-1, unit='ms', utc=True))
print("Janela real usada no notebook: [start_ms, end_ms)")

start_ms= 1783709100000 end_ms= 1785973500001
start (UTC) = 2026-07-10 18:45:00+00:00
end   (UTC) = 2026-08-05 23:45:00+00:00
Janela real usada no notebook: [start_ms, end_ms)


## 1. coverage by table

Tabelas grandes usam estimativas; não há COUNT completo nelas.

In [15]:
table_coverage = load_table_coverage(engine)
display(table_coverage)

,table_name,rows,first_ts,last_ts,count_kind
0,signal_point,0,1.783814e+12,1.785974e+12,estimate
1,signal_bucket,4936530,1.783709e+12,1.785974e+12,exact
2,operation_metrics,788556,1.783814e+12,1.785974e+12,exact
3,service_overview,385129,1.783814e+12,1.785974e+12,exact
4,prom_point,0,1.783815e+12,1.785974e+12,estimate
5,rpaas_bucket,52560,1.783814e+12,1.785974e+12,exact
6,ca_point,1399048,1.783814e+12,1.785974e+12,exact
7,tsuru_event,1903,1.783393e+12,1.785960e+12,exact
8,incidents,501,1.781265e+12,1.785968e+12,exact
9,incidents_platform,282,1.784154e+12,1.785968e+12,exact


## 2. Cobertura por signal_bucket.source

As épocas em milissegundos são convertidas para UTC.

In [16]:
source_coverage = load_source_coverage(engine)
display(source_coverage)

,source,rows,first_ts,last_ts
0,apollo,2933700,2026-07-10 19:00:00+00:00,2026-08-05 23:45:00+00:00
1,backstage,19725,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
2,cdn_live_errors,122004,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
3,cdn_vod_errors,22594,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
4,cloudarmor,16809,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
5,newrelic,204975,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
6,origin_live_errors,100948,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
7,prometheus,517136,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
8,redis,347087,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00
9,rpaas,97862,2026-07-10 18:45:00+00:00,2026-08-05 23:45:00+00:00


## 3. source x day heatmap

O heatmap mostra a cobertura diária por `source`. Cada célula corresponde a um dia em UTC e a cor representa quantas linhas apareceram naquele dia. Se a matriz vier vazia, a janela não cobre o período com dados ou o recorte ficou curto demais para agregação diária útil.

In [17]:
heatmap_start_ms = start_ms
heatmap_end_ms = end_ms
source_day = load_source_daily_coverage(engine, heatmap_start_ms, heatmap_end_ms)
display(source_day)
coverage_heatmap(source_day, x="day", y="source", value="rows").show()

,source,rows,day
0,apollo,17262,2026-07-10 00:00:00+00:00
1,apollo,80213,2026-07-11 00:00:00+00:00
2,apollo,80052,2026-07-12 00:00:00+00:00
3,apollo,79191,2026-07-13 00:00:00+00:00
4,apollo,79373,2026-07-14 00:00:00+00:00
...,...,...,...
346,youbora_player_live,7162,2026-08-01 00:00:00+00:00
347,youbora_player_live,5754,2026-08-02 00:00:00+00:00
348,youbora_player_live,5515,2026-08-03 00:00:00+00:00
349,youbora_player_live,5490,2026-08-04 00:00:00+00:00


## 4. Cadência para uma série selecionada

Cadência mede o intervalo entre observações consecutivas da mesma identidade `source/entity/metric`. Isso ajuda a entender se a série foi coletada de forma contínua, esporádica ou em rajadas.

Os valores de `series_start_ms` e `series_end_ms` agora seguem a janela real do dump. Os identificadores `series_source`, `series_entity` e `series_metric` ainda precisam ser escolhidos a partir de uma série existente no catálogo.

In [18]:
source_distribution = load_source_distribution(engine, start_ms, end_ms)
selected_source = source_distribution.iloc[0]["source"] if not source_distribution.empty else None
series_catalog = (
    list_series(engine, selected_source, start_ms, end_ms) if selected_source is not None else None
)

if series_catalog is not None and not series_catalog.empty:
    first_series = series_catalog.iloc[0]
    series_source = first_series["source"]
    series_entity = first_series["entity"]
    series_metric = first_series["metric"]
    series_start_ms = start_ms
    series_end_ms = end_ms
    cadence = load_series_cadence(
        engine, series_source, series_entity, series_metric, series_start_ms, series_end_ms
    )
    display(cadence)
    cadence_histogram(cadence).show()
else:
    print("Sem série disponível para calcular cadência na janela atual.")

,bts,previous_bts,cadence_s
0,2026-07-10 19:00:00+00:00,NaT,NaN
1,2026-07-10 20:00:00+00:00,2026-07-10 19:00:00+00:00,3600.0
2,2026-07-10 21:00:00+00:00,2026-07-10 20:00:00+00:00,3600.0
3,2026-07-10 22:00:00+00:00,2026-07-10 21:00:00+00:00,3600.0
4,2026-07-10 23:00:00+00:00,2026-07-10 22:00:00+00:00,3600.0
...,...,...,...
892,2026-08-05 22:45:00+00:00,2026-08-05 22:30:00+00:00,900.0
893,2026-08-05 23:00:00+00:00,2026-08-05 22:45:00+00:00,900.0
894,2026-08-05 23:15:00+00:00,2026-08-05 23:00:00+00:00,900.0
895,2026-08-05 23:30:00+00:00,2026-08-05 23:15:00+00:00,900.0


## 5. Resumo de nulos e desconhecidos

`missingness` resume quantos valores nulos existem por coluna no catálogo escolhido. O detalhe importante é que zero, nulo e ausência de bucket não são a mesma coisa: zero é um valor observado; nulo é um valor ausente; bucket ausente é falta de linha para aquele instante.

In [19]:
small_table = "signal_bucket"
small_columns = ["source", "entity", "metric"]
missingness = load_missingness(engine, small_table, small_columns)
display(missingness)
missingness_bar(missingness).show()

,table_name,column_name,null_count,row_count,null_pct
0,signal_bucket,source,0,4936530,0.0
1,signal_bucket,entity,0,4936530,0.0
2,signal_bucket,metric,0,4936530,0.0


## 6. Execuções dos coletores

Essa tabela explica como os coletores se comportaram. `collector` identifica o processo lógico; `started_at` e `finished_at` são horários de execução em UTC; `duration_s` mede tempo gasto; `rows_written` mostra produção efetiva; e `status` marca se a execução terminou (`complete`) ou ficou pela metade (`incomplete`).

In [20]:
collector_health = load_collector_health(engine)
display(collector_health)
collector_timeline(collector_health).show()

,collector,started_at,finished_at,duration_s,rows_written,status
0,apollo,2026-08-06 21:30:02.750926+00:00,2026-08-06 21:30:05.039375+00:00,2.288449,21,complete
1,apollo,2026-08-06 21:00:03.488504+00:00,2026-08-06 21:00:05.963364+00:00,2.474860,455,complete
2,apollo,2026-08-06 20:30:03.332909+00:00,2026-08-06 20:30:05.411711+00:00,2.078802,46,complete
3,apollo,2026-08-06 20:00:02.838221+00:00,2026-08-06 20:00:05.180551+00:00,2.342330,439,complete
4,apollo,2026-08-06 19:30:03.081137+00:00,2026-08-06 19:30:06.211435+00:00,3.130298,51,complete
...,...,...,...,...,...,...
19371,ca,NaT,NaT,NaN,3600,incomplete
19372,ca,NaT,NaT,NaN,3600,incomplete
19373,ca,NaT,NaT,NaN,3600,incomplete
19374,ca,NaT,NaT,NaN,3600,incomplete


## 7. Watermark e chegadas tardias

NPAW apresenta aproximadamente **28 horas** de atraso estrutural. Por isso, o watermark precisa preservar uma janela de releitura para chegadas tardias em vez de assumir que o dado mais recente já está completo. A ausência de tickets não é um rótulo negativo confiável: pode refletir falta de abertura, coleta ou cobertura, e não ausência do incidente.